# Self-Healing Feeder: hourly restoration study on Kaggle

Re-runs isolation and restoration for every fault zone at every feeder load level
(LoadMult 0.30 to 1.10), then maps each hour of one real year of PJM AEP load onto
those levels. Simulation only.

**Inputs (right-hand panel → Add Input), both needed:**
1. **The project code:** your private dataset `self-healing-feeder-code` (upload `self-healing-feeder-kaggle.zip`),
   or set `REPO_URL` to a public GitHub repo and turn **Internet: On**.
2. **The load data:** search **"Hourly Energy Consumption"** (by Rob Mulla / robikscube, CC0) and add it.
   The notebook finds `AEP_hourly.csv` by itself.

**Settings:** Accelerator **None**. Then **Save Version → Save & Run All**. About 1–3 minutes.
Results: the `hourly/` folder in the **Output** panel.

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/<your-username>/self-healing-feeder.git"  # leave as is when using the uploaded bundle
DATASET_DIR = "/kaggle/input"
YEAR = None        # None = latest full calendar year in AEP_hourly.csv

In [ ]:
# ---- Get the code into /kaggle/working (the only writable, saved folder) ----
import os, shutil, subprocess

WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

if "<your-username>" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
else:
    # find the uploaded project: the folder that contains run.py
    src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
    if src is None:
        # the dataset may hold the bundle as a .zip that Kaggle did not unpack
        import zipfile
        zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
        for z in zips:
            zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
        src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle")
                    if "run.py" in files), None)
    if src is None:
        raise SystemExit("Set REPO_URL to your GitHub repo, or attach the project folder as a Kaggle Dataset.")
    shutil.copytree(src, WORK)   # /kaggle/input is read-only, so work on a copy
    shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)

os.chdir(WORK)
print("Code in", WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# ---- Install the OpenDSS engine (numpy, pandas, scikit-learn, matplotlib, networkx are preinstalled) ----
# The Kaggle upload bundle ships the Linux wheels in wheels/, so no Internet is needed.
if os.path.isdir(os.path.join(WORK, "wheels")):
    get_ipython().run_line_magic("pip", "install -q --no-index --find-links wheels OpenDSSDirect.py==0.9.4")
else:
    get_ipython().run_line_magic("pip", "install -q OpenDSSDirect.py==0.9.4")

In [ ]:
# ---- Is the load data attached? ----
import sys
sys.path.insert(0, WORK)
from shf import config
print("Load data:", config.LOAD_SHAPE_CSV, "| found:", config.LOAD_SHAPE_CSV.exists())
if not config.LOAD_SHAPE_CSV.exists():
    raise SystemExit("Add Input -> search 'Hourly Energy Consumption' (robikscube) -> Add, then run again.")

In [ ]:
# ---- Tests for this study (about 10-20 s) ----
get_ipython().system('python -m pytest -q tests/test_shf.py -k "hourly or classify or load_shape or hours_map"')

In [ ]:
# ---- 1) Load shape  2) restoration at every level and every hour ----
get_ipython().system("python -u scripts/load_shape.py" + (f" --year {YEAR}" if YEAR else ""))
get_ipython().system("python -u scripts/hourly_restoration.py")

In [ ]:
# ---- Results ----
import json
import matplotlib.pyplot as plt
import pandas as pd

h = json.load(open("results/hourly/hourly.json"))
ls = h["load_shape"]
print(f"Year {ls['year']}: {ls['hours']} hours, LoadMult {ls['mult_min']:.2f} to 1.00 (mean {ls['mult_mean']:.2f})")
print(f"Healthy load restored by ties, mean over the year: {h['year']['pct_restored_mean']:.1%} "
      f"(at the fault hour), {h['year']['pct_restored_held_mean']:.1%} (held for {h['hold_hours']} h)")
print("Grid steps where MORE load restores MORE (decision within one regulator tap of 1.05 pu):",
      h["level_steps_with_more_restored"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, c in (("summer", "tab:red"), ("winter", "tab:blue")):
    d = h["typical_day"][name]
    ax[0].plot(d["mult"], color=c, label=name)
    ax[1].plot([100 * v for v in d["pct_restored"]], color=c, label=f"{name}, at fault hour")
    ax[1].plot([100 * v for v in d["pct_restored_held"]], color=c, ls="--", label=f"{name}, held {h['hold_hours']} h")
ax[0].set(title="Typical day: LoadMult", xlabel="hour of day")
ax[1].set(title="Healthy load restored by ties (%)", xlabel="hour of day")
for a in ax:
    a.legend(); a.grid(alpha=.3)
plt.show()
lv = h["by_level"]
plt.figure(figsize=(6, 3.5))
plt.plot(lv["level"], [100 * v for v in lv["pct_restored"]], marker=".")
plt.xlabel("LoadMult"); plt.ylabel("% restored"); plt.title("Restored vs. load level"); plt.grid(alpha=.3)
plt.show()

In [ ]:
# ---- Put the results at the top of the Output panel ----
OUT = "/kaggle/working/hourly"
os.chdir("/kaggle/working")
if os.path.exists(OUT):
    shutil.rmtree(OUT)
shutil.move(os.path.join(WORK, "results", "hourly"), OUT)
PROV_SRC = os.path.join(WORK, "results", "provenance")
PROV_DST = os.path.join(OUT, "provenance")
os.makedirs(PROV_DST, exist_ok=True)
for pf in ("load_shape.json", "hourly_restoration.json"):
    p = os.path.join(PROV_SRC, pf)
    if os.path.exists(p):
        shutil.copy2(p, os.path.join(PROV_DST, pf))
shutil.rmtree(WORK)
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f"{f:28s} {os.path.getsize(p) / 1e3:10.1f} kB")
shutil.make_archive("/kaggle/working/hourly", "zip", OUT)   # one file to download: hourly.zip
print('Download /kaggle/working/hourly.zip from the Output panel.')


## Bring the results home

In the **Output** panel (right side, `/kaggle/working`), download **`hourly.zip`**.
Unzip it into `results/hourly/` of the project (or hand the zip to Claude Code).
The report page and the health check read the files from there.
